In [ ]:
import os, sys, subprocess
from IPython import get_ipython
_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic('matplotlib', 'inline')

ROOT = os.getcwd()
for _p in (os.getcwd(), os.path.dirname(os.getcwd())):
    if os.path.exists(os.path.join(_p, 'exp', 'common.py')):
        ROOT = _p
        break
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from exp.common import suppress_tqdm, load_json
suppress_tqdm()

MODEL = 'llm/Llama-3.1-8B'
FINAL = os.path.join(ROOT, 'exp', 'results')

def ensure(exp, extra_args=(), seeds=('42',)):
    """Run the experiment once if its summary is missing, else load it.
    Tables/figures are rendered below by each exp's own print_table()/plot()."""
    d = os.path.join(FINAL, exp)
    summ = os.path.join(d, exp + '_summary.json')
    if os.path.exists(summ):
        print(f'[{exp}] existing -> {summ}')
        return load_json(summ)
    print(f'[{exp}] missing -> running full experiment...')
    argv = [sys.executable, '-m', 'exp.' + exp, '--model_path', MODEL,
            '--seeds'] + list(seeds) + ['--output_dir', d] + list(extra_args)
    print('  $', ' '.join(argv))
    env = dict(os.environ)
    env.pop('PYTHONPATH', None)
    env.update({'CUDA_VISIBLE_DEVICES': '0', 'TOKENIZERS_PARALLELISM': 'false'})
    r = subprocess.run(argv, cwd=ROOT, env=env)
    if r.returncode != 0:
        raise RuntimeError(f'{exp} failed (exit {r.returncode})')
    return load_json(summ)

In [ ]:
# exp1 — token-level edit robustness
from exp.exp1 import print_table, plot
RATES = [0.01, 0.03, 0.05, 0.07, 0.10]
s = ensure('exp1', ['--samples', '100', '--attack_rates'] + [str(r) for r in RATES])
print_table(s, RATES)
plot(s, RATES, show=True)

In [ ]:
# exp2 — real-system scene robustness
from exp.exp2 import print_table, plot, DEFAULT_SCENES
s = ensure('exp2', ['--samples', '100'])
print_table(s, DEFAULT_SCENES)
plot(s, DEFAULT_SCENES, [], show=True)

In [ ]:
# exp3 — capacity table (Catopter vs ARS)
from exp.exp3 import print_table
s = ensure('exp3', ['--samples', '100'])
print_table(s)

In [ ]:
# exp4 — encode/decode speed
from exp.exp4 import print_table
s = ensure('exp4', ['--samples', '100'])
print_table(s)

In [ ]:
# exp5 — capacity x robustness Pareto (T sweep)
from exp.exp5 import print_table, plot
s = ensure('exp5', ['--samples', '200', '--top_p', '1.0'])
print_table(s)
plot(s, show=True)

In [ ]:
# exp6 — soft-hash strategy table
from exp.exp6 import print_table, RATES
s = ensure('exp6', ['--samples', '200'])
print_table(s, RATES)

In [ ]:
# exp7 — cross-model availability
MODELS = ('llama=llm/Llama-3.1-8B qwen=llm/Qwen3.5-9B '
          'gemma=llm/gemma4-8B glm=llm/glm4-9B')
s = ensure('exp7', ['--samples', '200', '--models', MODELS])
print_table(s)